In [ ]:
# Step 1: Data Loading and Inspection (3 marks)
import pandas as pd

df = pd.read_csv('../data/q1_heart_disease.csv')

print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())

df.head()


### Step 1 Interpretation
- The dataset contains patient records with clinical features and a binary target (`heart_disease`).  
- We checked the shape (rows × columns), data types, and missing values.  
- Columns such as `resting_bp` and `cholesterol` show missing values, which we will handle in preprocessing.


In [1]:
# Step 2: Exploratory Data Analysis
import matplotlib.pyplot as plt
import seaborn as sns

# Target distribution
sns.countplot(x='heart_disease', data=df)
plt.title("Target Class Distribution")
plt.show()

# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

# Age distribution by target
sns.histplot(data=df, x='age', hue='heart_disease', multiple='stack')
plt.title("Age Distribution by Heart Disease")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

### Step 2 Interpretation
- The target distribution plot shows whether the dataset is balanced between patients with and without heart disease.  
- The correlation heatmap highlights relationships between numerical variables and the target. For example, `max_hr` and `oldpeak` may show stronger associations.  
- The age distribution plot suggests that older patients are more likely to have heart disease compared to younger ones.


In [ ]:
# Step 3: Data Preprocessing 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Median imputation for missing values
df['resting_bp'] = df['resting_bp'].fillna(df['resting_bp'].median())
df['cholesterol'] = df['cholesterol'].fillna(df['cholesterol'].median())

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

# Features and target
X = df_encoded.drop('heart_disease', axis=1)
y = df_encoded['heart_disease']

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

### Step 3 Interpretation
- Missing values were handled using **median imputation**, which is robust to outliers.  
- Categorical variables were converted into numeric form using **one-hot encoding**.  
- Numerical features were standardized with **StandardScaler** to ensure consistent ranges.  
- The dataset was split into training and test sets with stratification to preserve class balance.

In [ ]:
# Step 4: Model Training
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained successfully.")

### Step 4 Interpretation
- Three supervised learning models were trained: Decision Tree, Random Forest, and Gradient Boosting.  
- All models used `random_state=42` for reproducibility.  
- These models will be compared in the next step using evaluation metrics.

In [ ]:
# Step 5: Model Evaluation 
from sklearn.metrics import classification_report, confusion_matrix

for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"\n{name} Results:")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))

### Step 5 Interpretation
- Each model’s performance was evaluated using **confusion matrix**, **precision**, **recall**, and **F1-score**.  
- Precision measures correctness of positive predictions, recall measures sensitivity, and F1 balances both.  
- The best-performing model is identified based on these metrics, not just accuracy.  
- For example, if Gradient Boosting shows higher recall and F1, it may be preferred for detecting heart disease cases.

In [ ]:
# Step 6: Hyperparameter Tuning 
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid, cv=5, scoring='f1')
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

# Evaluate tuned model
y_pred_tuned = grid.best_estimator_.predict(X_test)
print("Tuned Model Report:\n", classification_report(y_test, y_pred_tuned))


### Step 6 Interpretation
- Hyperparameter tuning was performed on the Random Forest model using **GridSearchCV**.  
- The best parameters (e.g., number of trees, maximum depth) were identified.  
- The tuned model’s performance was compared against the baseline.  
- If the tuned model improves F1-score or recall, it is considered more effective for predicting heart disease.